# 第4回: Robomimic データセットの理解

このノートブックでは、ロボット操作のための模倣学習ベンチマークである **Robomimic** のデータセットについて学びます。
実際のデータセットの代わりにダミーデータを生成し、**LeRobot形式** でのデータ管理方法と、
PyTorchの`Dataset`クラスを用いたデータローディングを実装します。

## 1. Robomimicの概要

### Robomimicとは

[Robomimic](https://robomimic.github.io/) は、ロボット操作のための**模倣学習（Imitation Learning）ベンチマーク**です。
人間のデモンストレーションデータを用いて、ロボットにタスクを模倣させる学習手法を研究・評価するためのフレームワークです。

主な特徴:
- **標準化されたタスクセット**: 様々な難易度のロボット操作タスク
- **多様なデモンストレーションデータ**: 品質の異なる複数種類のデータセット
- **再現可能な評価基盤**: 公平なアルゴリズム比較が可能

### タスクの種類

Robomimicは以下の4つの代表的なタスクを提供しています:

| タスク | 説明 | 難易度 |
|--------|------|--------|
| **Lift** | テーブル上の立方体を持ち上げる | 易 |
| **Can** | 缶をビンに移動させる | 中 |
| **Square** | 正方形のナットをペグに通す | 難 |
| **Transport** | 2つのロボットアームで物体を搬送する | 最難 |

### データセットの種類: PH vs MH

Robomimicのデータセットは、デモンストレーションの収集方法によって大きく2種類に分かれます:

| 項目 | PH (Proficient-Human) | MH (Multi-Human) |
|------|----------------------|-------------------|
| **収集者** | 熟練した1人のオペレータ | 複数の異なるオペレータ |
| **品質** | 高品質・一貫性あり | 品質にばらつきあり |
| **軌道の特徴** | スムーズで効率的 | 多様だがノイズが多い |
| **データ数** | 200エピソード | 300エピソード |
| **用途** | 基本的な模倣学習の評価 | ロバスト性の評価 |

- **PH**: 熟練者が収集した高品質なデモンストレーション。軌道が安定しており、模倣学習アルゴリズムの基本性能を評価するのに適しています。
- **MH**: 異なるスキルレベルの複数人が収集したデモンストレーション。現実世界での利用場面に近く、アルゴリズムのロバスト性を評価できます。

## 2. データセットの取得

### 実際のRobomimicデータセット

実際のRobomimicデータセットは以下の方法で取得できます:

```bash
# robomimicのインストール
pip install robomimic

# データセットのダウンロード（例: lift タスクの PH データ）
python -m robomimic.scripts.download_datasets --tasks lift --dataset_types ph
```

データはHDF5形式（`.hdf5`）で保存され、以下のような構造を持ちます:

```
data.hdf5
├── data/
│   ├── demo_0/
│   │   ├── obs/
│   │   │   ├── agentview_image   # カメラ画像 (T, H, W, C)
│   │   │   ├── robot0_eef_pos    # エンドエフェクタ位置 (T, 3)
│   │   │   └── robot0_joint_pos  # 関節角度 (T, 7)
│   │   └── actions               # 行動 (T, 7)
│   ├── demo_1/
│   │   └── ...
│   └── ...
└── mask/
    ├── train/
    └── valid/
```

### 今回のアプローチ

今回はRobomimicの実データの代わりに、**ダミーデータをLeRobot形式で生成**して使用します。
これにより、大規模なデータのダウンロードなしに、データ処理パイプラインの構築を学べます。

### ダミーデータ生成関数

以下の関数で、Robomimicライクなダミーデータを**LeRobot形式**（numpy配列）で生成します。
- 状態・行動: 正弦波ベースのスムーズな軌道 + ノイズ
- 画像: 軌道に沿って移動する色付き円

In [ ]:
import numpy as np
import os
import json
import shutil
from pathlib import Path

def create_dummy_robomimic_dataset(
    task="lift",
    dataset_type="ph",
    num_episodes=50,
    episode_length=100,
    img_size=64,
    state_dim=7,
    action_dim=7,
    save_dir="./data"
):
    """
    Create dummy Robomimic dataset in LeRobot-style numpy format.

    Directory structure (LeRobot-style):
    save_dir/
    ├── meta/
    │   └── info.json              # Dataset metadata
    ├── observation.images.agentview/
    │   ├── episode_000000.npy     # (T, H, W, C)
    │   ├── episode_000001.npy
    │   └── ...
    ├── observation.state.npy      # (num_episodes, T, state_dim)
    ├── action.npy                 # (num_episodes, T, action_dim)
    └── episode_data_index.json    # Start/end indices per episode
    """
    save_dir = Path(save_dir)

    # ディレクトリの作成
    meta_dir = save_dir / "meta"
    img_dir = save_dir / "observation.images.agentview"
    meta_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    # ノイズレベルの設定（PHはスムーズ、MHはノイズが多い）
    if dataset_type == "ph":
        noise_scale = 0.02
        speed_variation = 0.1
    else:  # mh
        noise_scale = 0.15
        speed_variation = 0.5

    all_states = []
    all_actions = []
    episode_indices = {"from": [], "to": []}
    total_frames = 0

    for ep in range(num_episodes):
        # --- 滑らかな軌道の生成 ---
        t = np.linspace(0, 2 * np.pi, episode_length)

        # 各エピソードで異なる周波数・位相（MHはばらつきが大きい）
        freq = 1.0 + np.random.randn() * speed_variation
        phase = np.random.uniform(0, 2 * np.pi)

        # 状態: 正弦波の組み合わせ
        state = np.zeros((episode_length, state_dim), dtype=np.float32)
        for d in range(state_dim):
            state[:, d] = np.sin(freq * t + phase + d * np.pi / state_dim)
            state[:, d] += np.random.randn(episode_length) * noise_scale

        # 行動: 状態の時間差分 + ノイズ
        action = np.zeros((episode_length, action_dim), dtype=np.float32)
        action[:-1] = np.diff(state[:, :action_dim], axis=0)
        action[-1] = action[-2]
        action += np.random.randn(episode_length, action_dim) * noise_scale * 0.5

        all_states.append(state)
        all_actions.append(action)

        # エピソードのインデックス
        episode_indices["from"].append(total_frames)
        total_frames += episode_length
        episode_indices["to"].append(total_frames)

        # --- 画像の生成 ---
        images = np.zeros((episode_length, img_size, img_size, 3), dtype=np.uint8)
        # 背景色（タスクごとに異なる）
        bg_colors = {"lift": [40, 40, 60], "can": [40, 60, 40],
                     "square": [60, 40, 40], "transport": [50, 50, 50]}
        bg = bg_colors.get(task, [40, 40, 40])
        images[:] = bg

        # 軌道に沿って円を描画
        cx = ((state[:, 0] + 1) / 2 * (img_size - 10) + 5).astype(int)
        cy = ((state[:, 1] + 1) / 2 * (img_size - 10) + 5).astype(int)
        cx = np.clip(cx, 5, img_size - 5)
        cy = np.clip(cy, 5, img_size - 5)

        radius = 4
        for step in range(episode_length):
            # 円の描画（簡易版）
            for dx in range(-radius, radius + 1):
                for dy in range(-radius, radius + 1):
                    if dx * dx + dy * dy <= radius * radius:
                        px, py = cx[step] + dx, cy[step] + dy
                        if 0 <= px < img_size and 0 <= py < img_size:
                            # PH: 安定した色、MH: 色が変動
                            color_noise = 0 if dataset_type == "ph" else int(np.random.randn() * 20)
                            images[step, py, px] = [
                                min(255, max(0, 200 + color_noise)),
                                min(255, max(0, 100 + color_noise)),
                                min(255, max(0, 50 + color_noise)),
                            ]

        np.save(img_dir / f"episode_{ep:06d}.npy", images)

    # 状態・行動の保存
    all_states = np.stack(all_states)   # (num_episodes, T, state_dim)
    all_actions = np.stack(all_actions) # (num_episodes, T, action_dim)
    np.save(save_dir / "observation.state.npy", all_states)
    np.save(save_dir / "action.npy", all_actions)

    # メタデータの保存
    info = {
        "task": task,
        "dataset_type": dataset_type,
        "num_episodes": num_episodes,
        "episode_length": episode_length,
        "img_size": img_size,
        "state_dim": state_dim,
        "action_dim": action_dim,
        "total_frames": total_frames,
        "fps": 20,
        "robot_type": "Panda",
    }
    with open(meta_dir / "info.json", "w") as f:
        json.dump(info, f, indent=2)

    # エピソードインデックスの保存
    with open(save_dir / "episode_data_index.json", "w") as f:
        json.dump(episode_indices, f, indent=2)

    print(f"データセット生成完了: {save_dir}")
    print(f"  タスク: {task} ({dataset_type.upper()})")
    print(f"  エピソード数: {num_episodes}")
    print(f"  エピソード長: {episode_length}")
    print(f"  状態次元: {state_dim}, 行動次元: {action_dim}")
    print(f"  画像サイズ: {img_size}x{img_size}")
    print(f"  総フレーム数: {total_frames}")

    return save_dir

### PHデータとMHデータの両方を生成

In [ ]:
# PHデータの生成
ph_dir = create_dummy_robomimic_dataset(
    task="lift",
    dataset_type="ph",
    num_episodes=50,
    episode_length=100,
    save_dir="./data/robomimic_lift_ph"
)

print()

# MHデータの生成
mh_dir = create_dummy_robomimic_dataset(
    task="lift",
    dataset_type="mh",
    num_episodes=50,
    episode_length=100,
    save_dir="./data/robomimic_lift_mh"
)

### 生成されたディレクトリ構造の確認

In [ ]:
def show_tree(path, prefix="", max_files=5):
    """ディレクトリ構造を表示するユーティリティ"""
    path = Path(path)
    entries = sorted(path.iterdir())
    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]

    for d in dirs:
        print(f"{prefix}├── {d.name}/")
        show_tree(d, prefix + "│   ", max_files)

    for i, f in enumerate(files):
        if i < max_files:
            size = f.stat().st_size
            if size > 1024 * 1024:
                size_str = f"{size / 1024 / 1024:.1f} MB"
            elif size > 1024:
                size_str = f"{size / 1024:.1f} KB"
            else:
                size_str = f"{size} B"
            connector = "└──" if i == len(files) - 1 else "├──"
            print(f"{prefix}{connector} {f.name}  ({size_str})")
        elif i == max_files:
            print(f"{prefix}└── ... 他 {len(files) - max_files} ファイル")

print("=== PHデータセット ===")
show_tree("./data/robomimic_lift_ph")
print()
print("=== MHデータセット ===")
show_tree("./data/robomimic_lift_mh")

## 3. LeRobot形式のデータ構造

### LeRobotとは

[LeRobot](https://github.com/huggingface/lerobot) は、Hugging Faceが開発したロボット学習のためのオープンソースライブラリです。
模倣学習や強化学習で使うデータセットを統一的なフォーマットで管理するための仕組みを提供しています。

### LeRobot形式の特徴

| 要素 | 形式 | 説明 |
|------|------|------|
| **画像** | `observation.images.{camera}/episode_XXXXXX.npy` | エピソードごとに `(T, H, W, C)` の numpy 配列 |
| **状態** | `observation.state.npy` | 全エピソード分 `(N, T, state_dim)` の numpy 配列 |
| **行動** | `action.npy` | 全エピソード分 `(N, T, action_dim)` の numpy 配列 |
| **メタデータ** | `meta/info.json` | タスク名、次元数、FPS等の情報 |
| **インデックス** | `episode_data_index.json` | 各エピソードの開始・終了フレーム番号 |

### メタデータの確認

In [ ]:
import json

# メタデータの読み込みと表示
with open("./data/robomimic_lift_ph/meta/info.json") as f:
    info = json.load(f)

print("=== メタデータ (info.json) ===")
for key, value in info.items():
    print(f"  {key}: {value}")

### 各データの形状確認

In [ ]:
import numpy as np

# 状態データ
states = np.load("./data/robomimic_lift_ph/observation.state.npy")
print(f"状態データ shape: {states.shape}")
print(f"  -> (エピソード数, タイムステップ数, 状態次元)")

# 行動データ
actions = np.load("./data/robomimic_lift_ph/action.npy")
print(f"行動データ shape: {actions.shape}")
print(f"  -> (エピソード数, タイムステップ数, 行動次元)")

# 画像データ（1エピソード分）
images = np.load("./data/robomimic_lift_ph/observation.images.agentview/episode_000000.npy")
print(f"画像データ shape: {images.shape}")
print(f"  -> (タイムステップ数, 高さ, 幅, チャンネル)")
print(f"  画素値の範囲: [{images.min()}, {images.max()}]")

### 画像データの可視化

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False

images = np.load("./data/robomimic_lift_ph/observation.images.agentview/episode_000000.npy")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("PHデータ: エピソード0の画像サンプル", fontsize=14)

timesteps = np.linspace(0, len(images) - 1, 10, dtype=int)
for i, t in enumerate(timesteps):
    ax = axes[i // 5, i % 5]
    ax.imshow(images[t])
    ax.set_title(f"t={t}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Datasetクラスの実装

PyTorchの `torch.utils.data.Dataset` を継承して、LeRobot形式のRobomimicデータを読み込むクラスを実装します。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RobomimicDataset(Dataset):
    """
    LeRobot-style Robomimic dataset.

    Args:
        data_dir: データセットのディレクトリパス
        dataset_type: "ph" または "mh"
        task: "lift", "can", "square", "transport"
        img_size: 画像のリサイズ先（現在は元サイズのまま使用）
    """

    def __init__(self, data_dir, dataset_type="ph", task="lift", img_size=64):
        self.data_dir = Path(data_dir)
        self.dataset_type = dataset_type
        self.task = task
        self.img_size = img_size

        # メタデータの読み込み
        with open(self.data_dir / "meta" / "info.json") as f:
            self.info = json.load(f)

        # 状態・行動データの読み込み
        self.states = np.load(self.data_dir / "observation.state.npy")
        self.actions = np.load(self.data_dir / "action.npy")

        self.num_episodes = len(self.states)
        self.seq_len = self.states.shape[1]

        print(f"RobomimicDataset loaded: {task} ({dataset_type.upper()})")
        print(f"  エピソード数: {self.num_episodes}")
        print(f"  シーケンス長: {self.seq_len}")
        print(f"  状態次元: {self.states.shape[2]}")
        print(f"  行動次元: {self.actions.shape[2]}")

    def __len__(self):
        return self.num_episodes

    def __getitem__(self, idx):
        # このエピソードの画像を読み込む
        img_path = (
            self.data_dir / "observation.images.agentview"
            / f"episode_{idx:06d}.npy"
        )
        images = np.load(img_path)  # (T, H, W, C)

        # 正規化と軸の変換
        images = images.astype(np.float32) / 255.0
        images = images.transpose(0, 3, 1, 2)  # (T, C, H, W)

        states = self.states[idx].astype(np.float32)
        actions = self.actions[idx].astype(np.float32)

        return {
            "images": torch.from_numpy(images),
            "states": torch.from_numpy(states),
            "actions": torch.from_numpy(actions),
        }

    def get_episode_info(self, idx):
        """エピソードの統計情報を返す"""
        states = self.states[idx]
        actions = self.actions[idx]
        return {
            "episode_id": idx,
            "state_mean": states.mean(axis=0),
            "state_std": states.std(axis=0),
            "action_mean": actions.mean(axis=0),
            "action_std": actions.std(axis=0),
        }

### Datasetの使用例

In [ ]:
# PHデータセットの作成
ph_dataset = RobomimicDataset(
    data_dir="./data/robomimic_lift_ph",
    dataset_type="ph",
    task="lift"
)

print(f"\nデータセットサイズ: {len(ph_dataset)}")

# 1サンプル取得
sample = ph_dataset[0]
print(f"\n--- サンプルの形状 ---")
for key, val in sample.items():
    print(f"  {key}: {val.shape}, dtype={val.dtype}")

### DataLoaderとの連携

In [ ]:
# DataLoaderの作成
ph_loader = DataLoader(
    ph_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,  # ノートブック環境ではメインプロセスで実行
)

# 1バッチ取得
batch = next(iter(ph_loader))

print("=== バッチの形状 ===")
for key, val in batch.items():
    print(f"  {key}: {val.shape}")
    # images: (batch, T, C, H, W)
    # states: (batch, T, state_dim)
    # actions: (batch, T, action_dim)

### バッチデータの可視化

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle("バッチ内の4エピソードの画像（5タイムステップずつ）", fontsize=14)

for i in range(4):
    imgs = batch["images"][i]  # (T, C, H, W)
    timesteps = np.linspace(0, imgs.shape[0] - 1, 5, dtype=int)
    for j, t in enumerate(timesteps):
        img = imgs[t].permute(1, 2, 0).numpy()  # (H, W, C)
        img = np.clip(img, 0, 1)
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"ep={i}, t={t}")
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

## 5. PH/MHの切り替えデモ

`dataset_type` 引数を変えるだけで、PHとMHのデータセットを簡単に切り替えられます。
データの品質の違い（軌道のばらつき）を可視化して比較しましょう。

In [ ]:
# PHとMHの両方のデータセットを作成
ph_dataset = RobomimicDataset(
    data_dir="./data/robomimic_lift_ph",
    dataset_type="ph",
    task="lift"
)
print()
mh_dataset = RobomimicDataset(
    data_dir="./data/robomimic_lift_mh",
    dataset_type="mh",
    task="lift"
)

### 軌道の比較: PH vs MH

PHデータは軌道が安定している一方、MHデータは操作者ごとのばらつきが大きいことを確認します。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PH: 状態の軌道（次元0 vs 次元1）
ax = axes[0, 0]
ax.set_title("PH: 状態空間の軌道 (dim0 vs dim1)", fontsize=12)
for i in range(min(20, len(ph_dataset))):
    sample = ph_dataset[i]
    s = sample["states"].numpy()
    ax.plot(s[:, 0], s[:, 1], alpha=0.5, linewidth=0.8)
ax.set_xlabel("state[0]")
ax.set_ylabel("state[1]")
ax.grid(True, alpha=0.3)

# MH: 状態の軌道（次元0 vs 次元1）
ax = axes[0, 1]
ax.set_title("MH: 状態空間の軌道 (dim0 vs dim1)", fontsize=12)
for i in range(min(20, len(mh_dataset))):
    sample = mh_dataset[i]
    s = sample["states"].numpy()
    ax.plot(s[:, 0], s[:, 1], alpha=0.5, linewidth=0.8)
ax.set_xlabel("state[0]")
ax.set_ylabel("state[1]")
ax.grid(True, alpha=0.3)

# PH: 行動の時系列
ax = axes[1, 0]
ax.set_title("PH: 行動の時系列 (dim0)", fontsize=12)
for i in range(min(20, len(ph_dataset))):
    sample = ph_dataset[i]
    a = sample["actions"].numpy()
    ax.plot(a[:, 0], alpha=0.4, linewidth=0.8)
ax.set_xlabel("タイムステップ")
ax.set_ylabel("action[0]")
ax.grid(True, alpha=0.3)

# MH: 行動の時系列
ax = axes[1, 1]
ax.set_title("MH: 行動の時系列 (dim0)", fontsize=12)
for i in range(min(20, len(mh_dataset))):
    sample = mh_dataset[i]
    a = sample["actions"].numpy()
    ax.plot(a[:, 0], alpha=0.4, linewidth=0.8)
ax.set_xlabel("タイムステップ")
ax.set_ylabel("action[0]")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 統計量の比較

In [ ]:
# 各データセットの統計を計算
def compute_dataset_stats(dataset):
    all_states = []
    all_actions = []
    for i in range(len(dataset)):
        sample = dataset[i]
        all_states.append(sample["states"].numpy())
        all_actions.append(sample["actions"].numpy())

    states = np.stack(all_states)   # (N, T, D)
    actions = np.stack(all_actions)  # (N, T, D)

    # エピソード間のばらつき（各タイムステップでの標準偏差の平均）
    state_var = states.std(axis=0).mean()
    action_var = actions.std(axis=0).mean()

    # エピソード内のスムーズさ（隣接フレーム間の差分の大きさ）
    state_smoothness = np.abs(np.diff(states, axis=1)).mean()
    action_smoothness = np.abs(np.diff(actions, axis=1)).mean()

    return {
        "状態のエピソード間ばらつき": state_var,
        "行動のエピソード間ばらつき": action_var,
        "状態のスムーズさ (差分の平均)": state_smoothness,
        "行動のスムーズさ (差分の平均)": action_smoothness,
    }

ph_stats = compute_dataset_stats(ph_dataset)
mh_stats = compute_dataset_stats(mh_dataset)

print(f"{'指標':<35} {'PH':>10} {'MH':>10}")
print("-" * 57)
for key in ph_stats:
    print(f"{key:<35} {ph_stats[key]:>10.4f} {mh_stats[key]:>10.4f}")
print()
print("-> MHデータの方がばらつきが大きく、軌道のスムーズさも低いことが分かります。")

---
## 演習問題

### 演習1: ダミーデータセットの生成と形状確認

`create_dummy_robomimic_dataset` を使って `can` タスクのPHデータを生成し、
画像・状態・行動それぞれの形状（shape）を表示してください。

条件:
- タスク: `can`
- データセットタイプ: `ph`
- エピソード数: 30
- エピソード長: 80
- 画像サイズ: 48

In [ ]:
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
# canタスクのPHデータを生成
can_dir = create_dummy_robomimic_dataset(
    task="can",
    dataset_type="ph",
    num_episodes=30,
    episode_length=80,
    img_size=48,
    save_dir="./data/robomimic_can_ph"
)

# 各データの形状確認
states = np.load("./data/robomimic_can_ph/observation.state.npy")
actions = np.load("./data/robomimic_can_ph/action.npy")
images = np.load("./data/robomimic_can_ph/observation.images.agentview/episode_000000.npy")

print(f"状態データ: {states.shape}")   # (30, 80, 7)
print(f"行動データ: {actions.shape}")   # (30, 80, 7)
print(f"画像データ: {images.shape}")   # (80, 48, 48, 3)

# メタデータも確認
with open("./data/robomimic_can_ph/meta/info.json") as f:
    info = json.load(f)
print(f"\nメタデータ: {json.dumps(info, indent=2)}")

```

</details>

### 演習2: RobomimicDatasetとDataLoaderの活用

`RobomimicDataset` クラスを使ってPHデータセットを読み込み、
`DataLoader` でバッチサイズ8の1バッチを取得してください。

取得したバッチについて:
1. 各要素の形状を表示
2. バッチ内の最初のエピソードについて、最初と最後のフレーム画像を横に並べて表示

In [ ]:
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
# データセットとDataLoaderの作成
dataset = RobomimicDataset(
    data_dir="./data/robomimic_lift_ph",
    dataset_type="ph",
    task="lift"
)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# 1バッチ取得
batch = next(iter(loader))

# 形状の表示
print("=== バッチの形状 ===")
for key, val in batch.items():
    print(f"  {key}: {val.shape}")

# 最初のエピソードの最初と最後のフレームを表示
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle("エピソード0: 最初と最後のフレーム")

imgs = batch["images"][0]  # (T, C, H, W)
for i, (t, title) in enumerate([(0, "最初のフレーム"), (-1, "最後のフレーム")]):
    img = imgs[t].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(title)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

```

</details>

### 演習3: PH/MHデータの軌道比較可視化

PHデータとMHデータの **全次元** の状態軌道を比較するグラフを作成してください。

要件:
- 横軸: タイムステップ、縦軸: 状態値
- 7次元分を `plt.subplots(2, 4)` で表示（最後の1つは空欄でOK）
- PHは青系、MHは赤系で色分け
- 各次元について10エピソード分の軌道を重ねて描画
- タイトルにPH/MHの区別と次元番号を表示

In [ ]:
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
ph_dataset = RobomimicDataset("./data/robomimic_lift_ph", "ph", "lift")
mh_dataset = RobomimicDataset("./data/robomimic_lift_mh", "mh", "lift")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("PH (青) vs MH (赤): 全7次元の状態軌道比較", fontsize=14)

for d in range(7):
    ax = axes[d // 4, d % 4]

    # PH軌道
    for i in range(10):
        s = ph_dataset[i]["states"].numpy()
        ax.plot(s[:, d], color="steelblue", alpha=0.4, linewidth=0.8)

    # MH軌道
    for i in range(10):
        s = mh_dataset[i]["states"].numpy()
        ax.plot(s[:, d], color="indianred", alpha=0.4, linewidth=0.8)

    ax.set_title(f"dim {d}")
    ax.set_xlabel("タイムステップ")
    ax.set_ylabel(f"state[{d}]")
    ax.grid(True, alpha=0.3)

# 最後のサブプロットを非表示に
axes[1, 3].axis("off")

# 凡例（最後のサブプロットに配置）
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="steelblue", label="PH (高品質)"),
    Line2D([0], [0], color="indianred", label="MH (ばらつきあり)"),
]
axes[1, 3].legend(handles=legend_elements, loc="center", fontsize=14)

plt.tight_layout()
plt.show()

```

</details>

---
## まとめ

このノートブックでは以下を学びました:

1. **Robomimicの概要**: ロボット操作のための模倣学習ベンチマーク。Lift, Can, Square, Transportの4タスク
2. **PH/MHデータセットの違い**: PHは高品質で安定、MHは多様だがノイズが多い
3. **LeRobot形式のデータ構造**: numpy配列とJSONメタデータによる統一的なデータ管理
4. **PyTorch Datasetクラス**: LeRobot形式のデータを効率的に読み込むための実装
5. **データの可視化と比較**: 軌道や統計量を用いたPH/MHの品質差の分析

次回は、このデータセットを用いて実際に模倣学習モデルの学習を行います。